In [ ]:
"""
Simple & Effective Wheat Disease Detection with ConvNeXt

Simplified version keeping only the essentials:
✓ Multi-Scale Feature Fusion (main contribution)
✓ Clean, readable code (~400 lines vs 1700)
✓ Good performance with less complexity
✓ Easy to understand and modify
"""

import os
import json
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms, models
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import shutil
from PIL import Image
from itertools import product
from pathlib import Path
from typing import Dict, List

# =============================================================================
# Configuration
# =============================================================================
DATASET_DIR = '../../dataset'
SAVE_DIR = '../../saved_models_and_data'
SPLIT_OUTPUT_DIR = '../../dataset_split'

IMAGE_SIZE = (320, 320)  # Keep original size
BATCH_SIZE = 24
EPOCHS = 25  # Keep original epochs
LEARNING_RATE = 1e-4  # Keep original learning rate
WEIGHT_DECAY = 1e-4
EARLY_STOPPING_PATIENCE = 25  # Increased to allow all epochs to complete
LABEL_SMOOTHING = 0.1  # Added for better generalization

USE_MIXUP = True
MIXUP_ALPHA = 0.4  # Restored for better augmentation
USE_CUTMIX = True  # Add CutMix for additional augmentation
CUTMIX_ALPHA = 1.0
USE_TTA = True  # Test-time augmentation for final evaluation

MODEL_FILENAME = 'wheat_disease_convnext_model.pth'
MODEL_PATH = os.path.join(SAVE_DIR, MODEL_FILENAME)
LEGACY_MODEL_PATH = os.path.join(SAVE_DIR, 'best_model_simple.pth')

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(SPLIT_OUTPUT_DIR, exist_ok=True)

# =============================================================================
# Enhanced Focal Loss with Adaptive Gamma and Class-Specific Tuning
# =============================================================================
class FocalLoss(nn.Module):
    """Improved Focal Loss with adaptive gamma and progressive hard example boost"""
    def __init__(self, gamma=2.5, class_weights=None, device='cpu', 
                 adaptive_gamma=False, difficult_class_indices=None):
        super().__init__()
        self.gamma = gamma  # Balanced at 2.5 (was 2.8, too aggressive)
        self.adaptive_gamma = adaptive_gamma  # Disabled to prevent overfitting
        self.difficult_class_indices = difficult_class_indices or []
        self.class_weights = class_weights
        if class_weights is not None:
            self.class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)
    
    def forward(self, inputs, targets, label_smoothing=0.1):
        # Cross-entropy with class weights and label smoothing
        ce_loss = F.cross_entropy(
            inputs, targets, 
            reduction='none', 
            weight=self.class_weights,
            label_smoothing=label_smoothing
        )
        pt = torch.exp(-ce_loss)
        
        # Adaptive gamma: higher for difficult classes (30% boost)
        if self.adaptive_gamma and len(self.difficult_class_indices) > 0:
            gamma = self.gamma * torch.ones_like(targets, dtype=torch.float32)
            for idx in self.difficult_class_indices:
                gamma[targets == idx] = self.gamma * 1.3  # 30% higher gamma for difficult classes
        else:
            gamma = self.gamma
        
        # Standard focal loss
        focal_loss = (1 - pt) ** gamma * ce_loss
        
        # Enhanced hard example boost - progressive based on confidence
        probs = F.softmax(inputs, dim=1)
        target_probs = probs.gather(1, targets.unsqueeze(1)).squeeze(1)
        
        # Moderate hard example boost - less aggressive to prevent overfitting
        # Reduced boost to maintain balance between difficult and easy examples
        confidence_mask_low = target_probs < 0.3
        confidence_mask_medium = (target_probs >= 0.3) & (target_probs < 0.6)
        confidence_mask_high = target_probs >= 0.6
        
        hard_example_boost = torch.zeros_like(target_probs)
        hard_example_boost[confidence_mask_low] = 0.3 * (1 - target_probs[confidence_mask_low]) ** 2  # Reduced from 0.5
        hard_example_boost[confidence_mask_medium] = 0.2 * (1 - target_probs[confidence_mask_medium]) ** 2  # Reduced from 0.3
        hard_example_boost[confidence_mask_high] = 0.1 * (1 - target_probs[confidence_mask_high]) ** 2  # Kept same
        
        focal_loss = focal_loss * (1 + hard_example_boost)
        
        return focal_loss.mean()

# =============================================================================
# MixUp Augmentation
# =============================================================================
def mixup_data(x, y, alpha=0.4):
    """MixUp - mixes two images together"""
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    
    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(x.device)
    
    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def mixup_criterion(criterion, pred, y_a, y_b, lam):
    # For MixUp/CutMix, disable label smoothing since labels are already soft
    return lam * criterion(pred, y_a, label_smoothing=0.0) + (1 - lam) * criterion(pred, y_b, label_smoothing=0.0)

# =============================================================================
# CutMix Augmentation
# =============================================================================
def cutmix_data(x, y, alpha=1.0):
    """CutMix augmentation - cuts and pastes patches"""
    if alpha > 0:
        lam = np.random.beta(alpha, alpha)
    else:
        lam = 1
    
    batch_size = x.size()[0]
    index = torch.randperm(batch_size).to(x.device)
    
    # Generate random bounding box
    _, _, h, w = x.size()
    cut_rat = np.sqrt(1.0 - lam)
    cut_w = int(w * cut_rat)
    cut_h = int(h * cut_rat)
    
    cx = np.random.randint(w)
    cy = np.random.randint(h)
    
    bbx1 = np.clip(cx - cut_w // 2, 0, w)
    bby1 = np.clip(cy - cut_h // 2, 0, h)
    bbx2 = np.clip(cx + cut_w // 2, 0, w)
    bby2 = np.clip(cy + cut_h // 2, 0, h)
    
    x[:, :, bby1:bby2, bbx1:bbx2] = x[index, :, bby1:bby2, bbx1:bbx2]
    
    # Adjust lambda to match actual pixel ratio
    lam = 1 - ((bbx2 - bbx1) * (bby2 - bby1) / (w * h))
    
    y_a, y_b = y, y[index]
    return x, y_a, y_b, lam

# =============================================================================
# Test-Time Augmentation (TTA)
# =============================================================================
def test_time_augmentation(model, inputs, device, n_augments=5):
    """Apply TTA for more robust predictions - averages predictions from augmented versions"""
    model.eval()
    predictions = []
    
    # Original prediction
    with torch.no_grad():
        outputs = model(inputs)
        predictions.append(F.softmax(outputs, dim=1))
    
    # Augmented versions
    for _ in range(n_augments - 1):
        aug_inputs = inputs.clone()
        
        # Random augmentation
        aug_type = np.random.randint(3)
        
        if aug_type == 0:  # Horizontal flip
            aug_inputs = torch.flip(aug_inputs, [3])
        elif aug_type == 1:  # Vertical flip
            aug_inputs = torch.flip(aug_inputs, [2])
        elif aug_type == 2:  # Color jitter (simplified)
            brightness = 0.9 + 0.2 * np.random.rand()
            aug_inputs = aug_inputs * brightness
            aug_inputs = torch.clamp(aug_inputs, 0, 1)
        
        with torch.no_grad():
            outputs = model(aug_inputs)
            predictions.append(F.softmax(outputs, dim=1))
    
    # Average predictions
    avg_pred = torch.stack(predictions).mean(0)
    return avg_pred

# =============================================================================
# Multi-Scale Fusion Module (Main Innovation)
# =============================================================================
class MultiScaleFusion(nn.Module):
    """
    Multi-scale feature fusion with 3 branches
    This is our main contribution - captures disease features at different scales
    """
    def __init__(self, channels):
        super().__init__()
        
        # Three branches: 3x3, 5x5, 7x7 convolutions
        self.branch1 = nn.Sequential(
            nn.Conv2d(channels, channels, 3, padding=1, groups=channels//8),
            nn.Conv2d(channels, channels, 1),
            nn.BatchNorm2d(channels),
            nn.GELU()
        )
        
        self.branch2 = nn.Sequential(
            nn.Conv2d(channels, channels, 5, padding=2, groups=channels//8),
            nn.Conv2d(channels, channels, 1),
            nn.BatchNorm2d(channels),
            nn.GELU()
        )
        
        self.branch3 = nn.Sequential(
            nn.Conv2d(channels, channels, 7, padding=3, groups=channels//8),
            nn.Conv2d(channels, channels, 1),
            nn.BatchNorm2d(channels),
            nn.GELU()
        )
        
        # Fuse all branches
        self.fusion = nn.Sequential(
            nn.Conv2d(channels * 3, channels, 1),
            nn.BatchNorm2d(channels)
        )
    
    def forward(self, x):
        f1 = self.branch1(x)  # Fine details
        f2 = self.branch2(x)  # Medium patterns
        f3 = self.branch3(x)  # Large context
        
        # Concatenate and fuse
        concat = torch.cat([f1, f2, f3], dim=1)
        fused = self.fusion(concat)
        
        return fused

# =============================================================================
# Build Model
# =============================================================================
def build_model(num_classes):
    """Build ConvNeXt with Multi-Scale Fusion"""
    
    # Load pretrained ConvNeXt
    model = models.convnext_base(pretrained=True)
    in_features = model.classifier[2].in_features
    
    # Add our multi-scale fusion module
    model.fusion = MultiScaleFusion(in_features)
    
    # Enhanced classifier head for better performance
    model.classifier = nn.Sequential(
        nn.AdaptiveAvgPool2d(1),
        nn.Flatten(),
        nn.LayerNorm(in_features),
        nn.Dropout(0.2),  # Added early dropout
        nn.Linear(in_features, 768),  # Increased from 512
        nn.GELU(),
        nn.Dropout(0.3),
        nn.Linear(768, 384),  # Added intermediate layer
        nn.GELU(),
        nn.Dropout(0.2),
        nn.Linear(384, num_classes)
    )
    
    # Custom forward pass
    def forward(x):
        x = model.features(x)     # ConvNeXt backbone
        x = model.fusion(x)       # Our multi-scale fusion
        x = model.classifier(x)   # Classification head
        return x
    
    model.forward = forward
    return model

# =============================================================================
# Dataset & DataLoader
# =============================================================================
class WheatDiseaseDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = root_dir
        self.transform = transform
        self.classes = sorted(os.listdir(root_dir))
        self.class_to_idx = {cls: idx for idx, cls in enumerate(self.classes)}
        
        self.samples = []
        for cls in self.classes:
            class_dir = os.path.join(root_dir, cls)
            if not os.path.isdir(class_dir):
                continue
            for img_file in os.listdir(class_dir):
                if img_file.lower().endswith(('.png', '.jpg', '.jpeg')):
                    path = os.path.join(class_dir, img_file)
                    self.samples.append((path, self.class_to_idx[cls]))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        path, target = self.samples[idx]
        image = Image.open(path).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, target

# Transforms - Enhanced for better generalization
train_transform = transforms.Compose([
    transforms.Resize((int(IMAGE_SIZE[0] * 1.15), int(IMAGE_SIZE[1] * 1.15))),  # Slightly larger
    transforms.RandomCrop(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(35),  # Increased from 30
    transforms.RandomAffine(degrees=0, translate=(0.1, 0.1), scale=(0.9, 1.1)),  # Added
    transforms.ColorJitter(brightness=0.4, contrast=0.4, saturation=0.4, hue=0.1),  # Enhanced
    transforms.ToTensor(),  # Must convert to tensor before RandomErasing
    transforms.RandomErasing(p=0.1, scale=(0.02, 0.1)),  # Added for robustness (after ToTensor)
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

test_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

def get_dataloaders(batch_size: int = BATCH_SIZE):
    """Load or create data splits"""
    split_dirs = [os.path.join(SPLIT_OUTPUT_DIR, s) for s in ['train', 'val', 'test']]
    
    if all(os.path.isdir(d) for d in split_dirs):
        print("Loading existing data splits...")
        train_dataset = WheatDiseaseDataset(split_dirs[0], train_transform)
        val_dataset = WheatDiseaseDataset(split_dirs[1], test_transform)
        test_dataset = WheatDiseaseDataset(split_dirs[2], test_transform)
    else:
        print("Creating new data splits...")
        full_dataset = WheatDiseaseDataset(DATASET_DIR, train_transform)
        
        # Split data (70/15/15)
        indices = torch.randperm(len(full_dataset), generator=torch.Generator().manual_seed(42)).tolist()
        train_size = int(0.7 * len(full_dataset))
        val_size = int(0.15 * len(full_dataset))
        
        train_indices = indices[:train_size]
        val_indices = indices[train_size:train_size + val_size]
        test_indices = indices[train_size + val_size:]
        
        # Save splits to disk
        for split_name, split_indices in [('train', train_indices), ('val', val_indices), ('test', test_indices)]:
            for idx in split_indices:
                path, label = full_dataset.samples[idx]
                cls_name = full_dataset.classes[label]
                dest_dir = os.path.join(SPLIT_OUTPUT_DIR, split_name, cls_name)
                os.makedirs(dest_dir, exist_ok=True)
                shutil.copy(path, os.path.join(dest_dir, os.path.basename(path)))
        
        train_dataset = WheatDiseaseDataset(split_dirs[0], train_transform)
        val_dataset = WheatDiseaseDataset(split_dirs[1], test_transform)
        test_dataset = WheatDiseaseDataset(split_dirs[2], test_transform)
    
    # Weighted sampling with aggressive boosting for difficult classes
    targets = [s[1] for s in train_dataset.samples]
    class_counts = np.bincount(targets)
    class_weights = 1.0 / class_counts
    
    # OPTIMIZED BOOSTING for 95% accuracy
    class_names = train_dataset.classes
    print("\n" + "="*80)
    print("CLASS WEIGHT BOOSTING FOR DIFFICULT CLASSES")
    print("="*80)
    
    # OPTIMIZED BOOSTING for 95% accuracy
    # tan_spot: Increase boost to improve recall (model is struggling at 49% accuracy)
    # leaf_blight: Increase boost to improve recall (reduce false negatives)
    if 'tan_spot' in class_names:
        tan_idx = class_names.index('tan_spot')
        class_weights[tan_idx] *= 1.8  # Increased from 1.4x to 1.8x to improve learning (currently at 49% accuracy)
        print(f"✓ tan_spot: Boosted by 1.8x (index {tan_idx}, count: {class_counts[tan_idx]}) - Increased to improve recall (currently struggling at 49%)")
    
    if 'leaf_blight' in class_names:
        leaf_idx = class_names.index('leaf_blight')
        class_weights[leaf_idx] *= 2.6  # Increased from 2.4x to 2.6x for better recall (less false negatives)
        print(f"✓ leaf_blight: Boosted by 2.6x (index {leaf_idx}, count: {class_counts[leaf_idx]}) - Increased to improve recall")
    
    # Remove black_rust boost - it's performing well enough
    # if 'black_rust' in class_names:
    #     black_idx = class_names.index('black_rust')
    #     class_weights[black_idx] *= 1.2  # Very slight boost
    #     print(f"✓ black_rust: Boosted by 1.2x (index {black_idx})")
    
    # Normalize weights to maintain balance
    class_weights = class_weights / class_weights.sum() * len(class_weights)
    
    # Display class distribution
    print("\nClass Distribution:")
    for idx, cls_name in enumerate(class_names):
        count = class_counts[idx]
        weight = class_weights[idx]
        print(f"  {cls_name:25s}: {count:4d} samples, weight: {weight:.4f}")
    
    sample_weights = [class_weights[t] for t in targets]
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)
    print("="*80)
    
    # Create loaders with specified batch_size
    train_loader = DataLoader(train_dataset, batch_size=batch_size, sampler=sampler, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    
    print(f"\nData loaded: {len(train_dataset)} train, {len(val_dataset)} val, {len(test_dataset)} test")
    
    # Return class weights for loss function
    return train_loader, val_loader, test_loader, train_dataset.classes, class_weights

# =============================================================================
# Training Function
# =============================================================================
def train_model(model, device, train_loader, val_loader, num_epochs=EPOCHS, learning_rate=LEARNING_RATE, 
                class_weights=None, class_names=None, trial_name=""):
    """Training loop with enhanced loss for difficult classes"""
    
    # Identify difficult class indices for adaptive gamma
    difficult_class_indices = []
    if class_names:
        if 'tan_spot' in class_names:
            difficult_class_indices.append(class_names.index('tan_spot'))
        if 'leaf_blight' in class_names:
            difficult_class_indices.append(class_names.index('leaf_blight'))
    
    # Use optimized focal loss for 95% accuracy
    criterion = FocalLoss(
        gamma=2.7,  # Increased from 2.6 to 2.7 for better focus on hard examples
        class_weights=class_weights, 
        device=device,
        adaptive_gamma=False,  # Disabled to prevent over-aggressive focus
        difficult_class_indices=difficult_class_indices
    )
    optimizer = optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=WEIGHT_DECAY)
    
    # Learning rate schedule with warmup (2 epochs for 20 total epochs)
    def lr_lambda(epoch):
        warmup_epochs = 2
        if epoch < warmup_epochs:
            return (epoch + 1) / warmup_epochs  # Linear warmup
        else:
            # Cosine annealing after warmup
            progress = (epoch - warmup_epochs) / (num_epochs - warmup_epochs)
            return 0.5 * (1 + np.cos(np.pi * progress))
    
    scheduler = optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)
    
    best_acc = 0.0
    patience_counter = 0
    history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    
    print("\nStarting training...")
    for epoch in range(num_epochs):
        # Training phase
        model.train()
        train_loss, train_correct, train_total = 0, 0, 0
        
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            
            # Apply MixUp or CutMix augmentation (for better generalization)
            rand_val = np.random.rand()
            use_mixup = USE_MIXUP and rand_val > 0.6  # 40% MixUp
            use_cutmix = USE_CUTMIX and rand_val <= 0.6 and rand_val > 0.3  # 30% CutMix
            
            if use_mixup:
                inputs, labels_a, labels_b, lam = mixup_data(inputs, labels, MIXUP_ALPHA)
                outputs = model(inputs)
                loss = mixup_criterion(criterion, outputs, labels_a, labels_b, lam)
            elif use_cutmix:
                inputs, labels_a, labels_b, lam = cutmix_data(inputs, labels, CUTMIX_ALPHA)
                outputs = model(inputs)
                loss = mixup_criterion(criterion, outputs, labels_a, labels_b, lam)
            else:
                outputs = model(inputs)
                loss = criterion(outputs, labels, label_smoothing=LABEL_SMOOTHING)
            
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            
            train_loss += loss.item() * inputs.size(0)
            _, preds = torch.max(outputs, 1)
            train_correct += (preds == labels).sum().item()
            train_total += labels.size(0)
        
        # Validation phase with per-class tracking
        model.eval()
        val_loss, val_correct, val_total = 0, 0, 0
        val_per_class_correct = {}
        val_per_class_total = {}
        
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels, label_smoothing=LABEL_SMOOTHING)
                
                val_loss += loss.item() * inputs.size(0)
                _, preds = torch.max(outputs, 1)
                val_correct += (preds == labels).sum().item()
                val_total += labels.size(0)
                
                # Per-class tracking
                for label, pred in zip(labels.cpu().numpy(), preds.cpu().numpy()):
                    if label not in val_per_class_total:
                        val_per_class_correct[label] = 0
                        val_per_class_total[label] = 0
                    val_per_class_total[label] += 1
                    if label == pred:
                        val_per_class_correct[label] += 1
        
        # Calculate metrics
        train_loss /= train_total
        train_acc = train_correct / train_total
        val_loss /= val_total
        val_acc = val_correct / val_total
        
        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        
        scheduler.step()
        
        # Print main metrics
        print(f"Epoch {epoch+1:2d}/{num_epochs} - "
              f"Train Loss: {train_loss:.4f}, Acc: {train_acc:.4f} | "
              f"Val Loss: {val_loss:.4f}, Acc: {val_acc:.4f}")
        
        # Print tan_spot and leaf_blight accuracy (target classes)
        if len(val_per_class_total) > 0:
            # Find class indices (assuming class_names is available)
            # We'll get it from the loader's dataset
            try:
                class_names = val_loader.dataset.classes
                if 'tan_spot' in class_names:
                    tan_idx = class_names.index('tan_spot')
                    if tan_idx in val_per_class_total and val_per_class_total[tan_idx] > 0:
                        tan_acc = val_per_class_correct[tan_idx] / val_per_class_total[tan_idx]
                        print(f"  → tan_spot Acc: {tan_acc:.4f} ({val_per_class_correct[tan_idx]}/{val_per_class_total[tan_idx]})")
                
                if 'leaf_blight' in class_names:
                    leaf_idx = class_names.index('leaf_blight')
                    if leaf_idx in val_per_class_total and val_per_class_total[leaf_idx] > 0:
                        leaf_acc = val_per_class_correct[leaf_idx] / val_per_class_total[leaf_idx]
                        print(f"  → leaf_blight Acc: {leaf_acc:.4f} ({val_per_class_correct[leaf_idx]}/{val_per_class_total[leaf_idx]})")
            except:
                pass  # Skip if class names not available
        
        # Save best model
        if val_acc > best_acc:
            best_acc = val_acc
            if trial_name:
                # Save with trial name for grid search
                best_model_path = os.path.join(
                    SAVE_DIR, f"best_msconvnext_model_{trial_name.replace(' ', '_')}.pth"
                )
                torch.save(model.state_dict(), best_model_path)
            else:
                # Save with default names for regular training
                torch.save(model.state_dict(), MODEL_PATH)
                torch.save(model.state_dict(), LEGACY_MODEL_PATH)
            print(f"  → New best! Val Acc: {val_acc:.4f}")
            patience_counter = 0
        else:
            patience_counter += 1
        
        # Early stopping
        if patience_counter >= EARLY_STOPPING_PATIENCE:
            print(f"\nEarly stopping at epoch {epoch+1}")
            break
    
    print(f"\nTraining complete! Best Val Acc: {best_acc:.4f}")
    return model, history, best_acc

# =============================================================================
# Evaluation & Visualization
# =============================================================================
def evaluate_and_visualize(model, device, test_loader, class_names, history):
    """Evaluate model and create visualizations"""
    
    # Load best model (prefer new filename, fallback to legacy)
    checkpoint_path = MODEL_PATH if os.path.exists(MODEL_PATH) else LEGACY_MODEL_PATH
    if checkpoint_path == LEGACY_MODEL_PATH and not os.path.exists(checkpoint_path):
        raise FileNotFoundError(f"No checkpoint found at {MODEL_PATH} or {LEGACY_MODEL_PATH}")
    if checkpoint_path == LEGACY_MODEL_PATH:
        print(f"⚠️ Using legacy checkpoint: {LEGACY_MODEL_PATH}")
    model.load_state_dict(torch.load(checkpoint_path))
    model.eval()
    
    # Evaluate on test set WITH TTA
    y_true, y_pred = [], []
    print("\nEvaluating on test set...")
    
    if USE_TTA:
        print("Using Test-Time Augmentation (TTA) for more robust predictions...")
    
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            
            if USE_TTA:
                # Use TTA for more robust predictions
                outputs = test_time_augmentation(model, inputs, device, n_augments=7)  # Increased from 5 to 7 for better accuracy
                _, preds = torch.max(outputs, 1)
            else:
                outputs = model(inputs)
                _, preds = torch.max(outputs, 1)
            
            y_true.extend(labels.numpy())
            y_pred.extend(preds.cpu().numpy())
    
    test_acc = np.mean(np.array(y_true) == np.array(y_pred))
    print(f"\n✓ Test Accuracy: {test_acc*100:.2f}%")
    
    # Classification report
    print("\nClassification Report:")
    print(classification_report(y_true, y_pred, target_names=class_names, digits=3))
    
    # Training curves
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    ax1.plot(history['train_loss'], label='Train', marker='o')
    ax1.plot(history['val_loss'], label='Val', marker='s')
    ax1.set_xlabel('Epoch', fontsize=12)
    ax1.set_ylabel('Loss', fontsize=12)
    ax1.set_title('Training & Validation Loss', fontsize=14, fontweight='bold')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    ax2.plot([a*100 for a in history['train_acc']], label='Train', marker='o')
    ax2.plot([a*100 for a in history['val_acc']], label='Val', marker='s')
    ax2.set_xlabel('Epoch', fontsize=12)
    ax2.set_ylabel('Accuracy (%)', fontsize=12)
    ax2.set_title('Training & Validation Accuracy', fontsize=14, fontweight='bold')
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, 'training_curves_simple.png'), dpi=200)
    print("✓ Training curves saved")
    plt.show()
    
    # Confusion matrix
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(12, 10))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
                xticklabels=class_names, yticklabels=class_names, 
                cbar_kws={'label': 'Count'})
    plt.xlabel('Predicted', fontsize=12, fontweight='bold')
    plt.ylabel('True', fontsize=12, fontweight='bold')
    plt.title(f'Confusion Matrix (Test Acc: {test_acc*100:.2f}%)', 
              fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(SAVE_DIR, 'confusion_matrix_simple.png'), dpi=200)
    print("✓ Confusion matrix saved")
    plt.show()
    
    print(f"\n✓ All results saved to: {SAVE_DIR}")

# =============================================================================
# Evaluation Function for Grid Search
# =============================================================================
def evaluate_on_test(model, device, test_loader, class_labels):
    """Evaluate model on test set and return metrics"""
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs = inputs.to(device)
            labels = labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            y_true.extend(labels.cpu().numpy())
            y_pred.extend(preds.cpu().numpy())

    conf_matrix = confusion_matrix(y_true, y_pred)
    report = classification_report(
        y_true, y_pred, target_names=class_labels, digits=4, output_dict=True
    )
    return conf_matrix, report

# =============================================================================
# Grid Search Function
# =============================================================================
def run_grid_search(
    learning_rates: List[float],
    batch_sizes: List[int],
    num_epochs: int = EPOCHS,
):
    """Run grid search over learning rates and batch sizes"""
    results: List[Dict] = []
    trial_id = 0

    for lr, batch_size in product(learning_rates, batch_sizes):
        trial_id += 1
        trial_name = f"trial{trial_id}_lr{lr}_bs{batch_size}"
        print("\n" + "=" * 80)
        print(f"Starting {trial_name}")
        print("=" * 80)

        train_loader, val_loader, test_loader, class_labels, class_weights = get_dataloaders(
            batch_size=batch_size
        )
        device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        model = build_model(len(class_labels)).to(device)

        model, train_log, best_val_acc = train_model(
            model=model,
            device=device,
            train_loader=train_loader,
            val_loader=val_loader,
            num_epochs=num_epochs,
            learning_rate=lr,
            class_weights=class_weights,
            class_names=class_labels,
            trial_name=trial_name,
        )

        # Load best weights for this trial
        best_model_path = os.path.join(
            SAVE_DIR, f"best_msconvnext_model_{trial_name.replace(' ', '_')}.pth"
        )
        if os.path.exists(best_model_path):
            model.load_state_dict(
                torch.load(best_model_path, map_location=device)
            )

        conf_matrix, report = evaluate_on_test(
            model, device, test_loader, class_labels
        )

        # Compute overall test accuracy from report
        test_acc = report["accuracy"]

        trial_result = {
            "trial_name": trial_name,
            "learning_rate": lr,
            "batch_size": batch_size,
            "best_val_acc": best_val_acc,
            "test_acc": test_acc,
        }
        results.append(trial_result)

        # Save per-trial logs
        trial_dir = Path(SAVE_DIR) / "grid_search_msconvnext" / trial_name
        trial_dir.mkdir(parents=True, exist_ok=True)
        with open(trial_dir / "train_log.json", "w") as f:
            json.dump(train_log, f, indent=2)
        with open(trial_dir / "classification_report.json", "w") as f:
            json.dump(report, f, indent=2)
        np.save(trial_dir / "confusion_matrix.npy", conf_matrix)

    # Sort results by validation accuracy
    results_sorted = sorted(results, key=lambda x: x["best_val_acc"], reverse=True)
    leaderboard_path = Path(SAVE_DIR) / "grid_search_msconvnext" / "leaderboard.json"
    leaderboard_path.parent.mkdir(parents=True, exist_ok=True)
    with open(leaderboard_path, "w") as f:
        json.dump(results_sorted, f, indent=2)

    print("\nGrid search completed. Top configurations:")
    for r in results_sorted[:5]:
        print(
            f"{r['trial_name']} | lr={r['learning_rate']} | "
            f"bs={r['batch_size']} | best_val_acc={r['best_val_acc']:.4f} | "
            f"test_acc={r['test_acc']:.4f}"
        )

# =============================================================================
# Main
# =============================================================================
if __name__ == '__main__':
    print("="*80)
    print("MSConvNeXt GRID SEARCH - WHEAT DISEASE DETECTION")
    print("="*80)
    
    # Grid search configuration
    lr_list = [1e-4, 5e-5]
    batch_size_list = [16, 24, 32]
    
    print(f"\nGrid Search Configuration:")
    print(f"  Learning Rates: {lr_list}")
    print(f"  Batch Sizes: {batch_size_list}")
    print(f"  Total Trials: {len(lr_list) * len(batch_size_list)}")
    print("="*80 + "\n")
    
    # Run grid search
    run_grid_search(learning_rates=lr_list, batch_sizes=batch_size_list, num_epochs=EPOCHS)
    
    print("\n" + "="*80)
    print("✓ GRID SEARCH COMPLETE!")
    print("="*80)

